<a href="https://colab.research.google.com/github/Sakinat-Folorunso/OOU_CSC309_Artificial_Intelligence/blob/main/notebooks/CSC309_Week08_NLP2_Sentiment_Spelling_MT_Student_Centred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 309 – Artificial Intelligence
**Week 8 Practical (PH):** NLP II — sentiment, spelling correction, machine translation and speech recognition

**Instructor:** Dr Sakinat Folorunso · Associate Professor of AI Systems · Department of Computer Sciences, Olabisi Onabanjo University (OOU), Ago-Iwoye, Ogun State, Nigeria  
**Course:** CSC 309 (2 units; LH 15, PH 45) · **Mode:** student-centred, hands-on, Google Colab  
**CCMAS learning outcomes covered:** LO7

> 🎓 **Instructor note** – this notebook is the *STUDENT (TODO cells are stubs)* version.


## How to use this notebook
1. Fill in the **Group Log**, then do the **Do Now** with your partner.
2. Run the **Setup** cell once (*Runtime → Run all* is safe: nothing here crashes if a TODO is still empty).
3. Work top to bottom. Only edit cells marked **`# TODO(Student)`**.
4. Each task ends with an **auto-check** (✅/❌). Keep going until it is ✅.
5. Finish with the **Reflection**. Fast finishers: try the **Extensions**.

6. **Read the comments.** Every line of code has a plain-English comment above it. Read the comment first, predict what the line does, then run the cell and check your prediction.
7. Hand in the **Assignment** (marks and deadline are in the notebook) after the lab.

*Every random draw uses `SEED = 42`, so your numbers should match your classmates' numbers.*


## Lab tasks at a glance

| Part | Lab task(s) you must complete |
|---|---|
| Part A — Sentiment analysis (opinion mining) | Task A3 — Adapt the lexicon to Pidgin |
| Part B — Spelling correction (word-processor feature) | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part C — Machine translation (English → Yorùbá, toy) | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part D — Speech recognition (isolated words) with spectrograms and DTW | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| **Assignment** | see **📝 Assignment / deliverables** near the end |
| **Reflection** | short written reflection at the end (5–7 sentences) |
| *Extensions* | optional, for fast finishers |

In [ ]:
#@title 🧑🏽‍🤝‍🧑🏾 Group Log (fill before you start)
# Form field: type your group members' names here
group_members = "Type names here"   #@param {type:"string"}
# Form field: assign roles (driver, navigator, recorder, presenter)
roles_notes   = "Driver / Navigator / Recorder / Presenter"  #@param {type:"string"}
# Print them so your notebook records who did what
print("👥 Group:", group_members); print("📝 Roles:", roles_notes)


In [ ]:
#@title 🔧 Setup (run once) — seed, logging, helpers
# Standard-library modules used by every lab (os, re, math, random, logging, time ...)
import sys, os, re, math, random, logging, subprocess, warnings, time, itertools
# NumPy: fast arrays and random numbers
import numpy as np
# One seed for the whole course so every run gives the same numbers
SEED = 42                                    # one seed for the whole course → reproducible results
# Seed both Python's and NumPy's random generators
random.seed(SEED); np.random.seed(SEED)
# Configure logging so messages print as "LEVEL | message"
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
# A named logger for this course
log = logging.getLogger("csc309")
# Hide noisy library warnings so output stays readable
warnings.filterwarnings("ignore")

def ensure(pip_name, import_name=None):
    """Import a package; pip-install it first if it is missing (Colab-friendly)."""
    # Try to import the library
    try:
        # Return the imported module
        return __import__(import_name or pip_name)
    # If it is not installed ...
    except ImportError:
        # ... tell the student we are installing it
        log.info("installing %s ...", pip_name)
        # ... and install it with pip
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        # Return the imported module
        return __import__(import_name or pip_name)

# List that stores every check result
_CHECKS = []
def check(name, condition, hint=""):
    """Auto-check. Never raises, so 'Run all' always finishes. Pass a lambda to catch your own bugs."""
    # Try to import the library
    try:
        # Run the check: call it if it is a function, otherwise use the value
        ok = bool(condition() if callable(condition) else condition)
        # Default: no error message
        err = ""
    # If the student's code crashes, we report instead of stopping the notebook
    except Exception as e:                    # student code may crash; report instead of stopping
        # Mark as failed and keep the error message
        ok, err = False, f"{type(e).__name__}: {e}"
    # Store the result
    _CHECKS.append((name, ok))
    # Print a ✅ or ❌ line with a hint on failure
    print(("✅ PASS       " if ok else "❌ TRY AGAIN  ") + name + ("" if ok else f"   → {err or hint}"))
    # Return whether the check passed
    return ok

# Print how many checks passed
def checks_summary():
    # Count the passed checks
    passed = sum(ok for _, ok in _CHECKS)
    # Print the total
    print(f"\nChecks passed: {passed}/{len(_CHECKS)}")

def mean_ci(x, conf=0.95):
    """Mean, sample std and a two-sided t-based confidence interval."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert to a float array and count the observations
    x = np.asarray(x, dtype=float); n = len(x)
    # Mean and sample standard deviation (n-1 divisor)
    m = x.mean(); s = x.std(ddof=1) if n > 1 else 0.0
    # Half-width of the 95% confidence interval from the t-distribution
    h = stats.t.ppf((1 + conf) / 2, n - 1) * s / math.sqrt(n) if n > 1 else 0.0
    # Return mean, SD, lower and upper confidence limits
    return m, s, m - h, m + h

# Print a one-line statistical summary
def report(label, x):
    # Compute the statistics
    m, s, lo, hi = mean_ci(x)
    # Print mean, SD, CI and n
    print(f"{label:<28} mean={m:8.2f}  sd={s:7.2f}  95% CI=[{lo:.2f}, {hi:.2f}]  (n={len(x)})")

def paired_wilcoxon(a, b):
    """Two-sided Wilcoxon signed-rank test for paired samples (same seeds/instances)."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert both samples to arrays
    a, b = np.asarray(a, float), np.asarray(b, float)
    # If the two samples are identical the test is undefined
    if np.allclose(a, b):
        # Report p = 1 (no difference)
        return 1.0
    # Paired Wilcoxon signed-rank test p-value
    return float(stats.wilcoxon(a, b).pvalue)

# Setup finished
print("✅ Setup complete — seed =", SEED)


### Learning objectives (with Bloom level)
| # | By the end of this lab you can… | Bloom | CCMAS LO |
|---|---|---|---|
| 1 | build **rule-based** and **ML** sentiment classifiers and compare them | Apply / Evaluate | LO7 |
| 2 | measure whether a model works **equally well for different language varieties** (English vs Nigerian Pidgin) | Evaluate | LO7 |
| 3 | implement **edit distance** and a noisy-channel **spelling corrector** | Apply | LO7 |
| 4 | explain **machine translation** (word order!) and score it with **BLEU** | Understand / Apply | LO7 |
| 5 | build an isolated-word **speech recogniser** with spectrogram features and **dynamic time warping** | Apply / Create | LO7 |

**Formulas.**
*Levenshtein distance* $d(i,j)=\min\{\,d(i-1,j)+1,\; d(i,j-1)+1,\; d(i-1,j-1)+[a_i\neq b_j]\,\}$ with $d(i,0)=i,\;d(0,j)=j$.
*Noisy-channel spelling correction:* $\hat w=\arg\max_{w\in V} P(x\mid w)\,P(w)$ (error model × word frequency).
*BLEU-$N$:* $\mathrm{BP}\cdot\exp\!\big(\sum_{n=1}^{N}\tfrac1N\log p_n\big)$, $\mathrm{BP}=\min\!\big(1,e^{\,1-r/c}\big)$ ($c$ = candidate length, $r$ = reference length).
*DTW:* $D(i,j)=c(i,j)+\min\{D(i-1,j),D(i,j-1),D(i-1,j-1)\}$.

**Do Now (3 min).** Send a message in Pidgin or Yorùbá to a classmate. Which parts would Google Translate or an English sentiment tool get wrong? Why?

## Part A — Sentiment analysis (opinion mining)
**A1. Rule-based:** sum word polarities from a *lexicon*; handle **negation** ("not good") and **intensifiers** ("very good"). Complete the scorer.

In [ ]:
#@title 📖 Lexicon + scorer (TODO: negation and intensifiers)
# Sentiment lexicon: word -> strength (positive words score +1 to +3)
LEXICON = {"good":1,"great":2,"excellent":3,"amazing":3,"love":2,"like":1,"nice":1,"happy":2,"helpful":2,"fast":1,"clean":1,"easy":1,"best":3,"wonderful":3,
           "bad":-1,"terrible":-3,"awful":-3,"hate":-2,"slow":-1,"dirty":-2,"poor":-2,"worst":-3,"useless":-2,"broken":-2,"disappointing":-2,"scam":-3,"rude":-2,"late":-1}
# Words that flip the meaning of the next words
NEGATORS = {"not", "no", "never", "cannot", "dont", "don't", "isn't", "wasn't", "didn't"}
# Intensifiers that multiply a word's strength
BOOST = {"very": 1.5, "really": 1.5, "extremely": 2.0}

# Score a text by adding up the strengths of its sentiment words
def lex_score(text):
    # Lower-case, split into words, start the total at 0
    toks = re.findall(r"[a-z']+", text.lower()); total = 0.0
    # Look at each word with its position
    for i, w in enumerate(toks):
        # Is the word in the lexicon?
        if w in LEXICON:
            # Its base strength
            s = LEXICON[w]
            # TODO(Student): (1) multiply by BOOST[toks[i-1]] if the previous word is an intensifier
            #                (2) flip the sign if a NEGATOR occurs in the 3 words before w
            # Add it to the running total
            total += s
    # Positive total = positive review; negative total = negative review
    return total

In [ ]:
#@title ✅ Auto-check for A1
check("'the lecture was good' is positive",       lambda: lex_score("the lecture was good") > 0)
check("negation: 'not good' is negative",          lambda: lex_score("it was not good") < 0)
check("intensifier: 'very good' > 'good'",         lambda: lex_score("very good") > lex_score("good"))
check("negated negative: 'not bad' is positive",   lambda: lex_score("it is not bad") > 0)

**A2. Compare three methods on movie reviews** (2,000 labelled reviews from NLTK; if the download is blocked a tiny fallback set is used and the numbers are *not* meaningful): your lexicon, **VADER** (a rule-based tool with a large lexicon), and a **TF-IDF + LinearSVC** machine-learning model (5-fold CV). Report accuracy with 95% Wilson CIs.

In [ ]:
#@title 🎬 Lexicon vs VADER vs ML
# Install/import NLTK if needed
nltk = ensure("nltk")
# Load the movie-review data set
def load_reviews():
    # Try the real corpus first
    try:
        # Download it quietly
        nltk.download("movie_reviews", quiet=True)
        # Import the corpus reader
        from nltk.corpus import movie_reviews
        # List of review file ids
        ids = movie_reviews.fileids()
        # Return texts, labels (1 = positive) and the data source
        return [movie_reviews.raw(f) for f in ids], np.array([int(movie_reviews.categories(f)[0] == "pos") for f in ids]), "NLTK movie_reviews (real, n=2000)"
    # If it is unavailable ...
    except Exception as e:
        # ... log a warning and use a tiny fallback
        log.warning("movie_reviews unavailable (%s) → tiny fallback", type(e).__name__)
        # Twelve tiny example sentences (fallback only)
        t = ["great fast and helpful", "excellent and wonderful", "i love it", "very good value", "nice and clean", "best ever",
             "terrible and slow", "awful and rude", "i hate it", "very poor service", "dirty and broken", "worst ever"]
        # Return the sentences and their labels
        return t, np.array([1]*6 + [0]*6), "FALLBACK 12 sentences (not meaningful)"

# Load the reviews and print where they came from
reviews, labels, RSRC = load_reviews(); print(RSRC)
# SciPy for confidence intervals
from scipy import stats
# Accuracy with a 95% Wilson confidence interval
def acc_ci(correct):
    # k correct out of n; Wilson interval
    k, n = int(np.sum(correct)), len(correct); ci = stats.binomtest(k, n).proportion_ci(0.95, method="wilson")
    # Return accuracy, lower and upper bound
    return k / n, ci.low, ci.high

# Predict positive if the lexicon score is above 0
pred_lex = np.array([1 if lex_score(t) > 0 else 0 for t in reviews])               # ties → negative (score 0)
# Print accuracy with CI
print("lexicon       acc = %.3f  [%.3f, %.3f]" % acc_ci(pred_lex == labels))
# Try the real corpus first
try:
    # Download the VADER lexicon
    nltk.download("vader_lexicon", quiet=True)
    # Import the VADER analyser
    from nltk.sentiment import SentimentIntensityAnalyzer
    # Create the analyser
    sia = SentimentIntensityAnalyzer()
    # Positive if the compound score is at least 0.05
    pred_vader = np.array([1 if sia.polarity_scores(t)["compound"] >= 0.05 else 0 for t in reviews])
    # Print accuracy with CI
    print("VADER         acc = %.3f  [%.3f, %.3f]" % acc_ci(pred_vader == labels))
# If it is unavailable ...
except Exception as e:
    # VADER could not be loaded: log and carry on
    log.warning("VADER unavailable (%s)", type(e).__name__)

# Text to TF-IDF features
from sklearn.feature_extraction.text import TfidfVectorizer
# Linear support vector machine
from sklearn.svm import LinearSVC
# Pipeline helper
from sklearn.pipeline import make_pipeline
# Cross-validated predictions
from sklearn.model_selection import cross_val_predict, StratifiedKFold
# Number of folds (fewer for tiny data)
n_splits = 5 if len(reviews) > 100 else 3
# Predict with a TF-IDF + SVM pipeline in cross-validation
pred_ml = cross_val_predict(make_pipeline(TfidfVectorizer(min_df=2 if len(reviews) > 100 else 1, sublinear_tf=True), LinearSVC()), reviews, labels,
                            cv=StratifiedKFold(n_splits, shuffle=True, random_state=SEED))
# Print accuracy with CI and the number of folds
print("TF-IDF+SVM    acc = %.3f  [%.3f, %.3f]   (%d-fold CV)" % (*acc_ci(pred_ml == labels), n_splits))
check("ML model beats the hand-made lexicon on the movie reviews", lambda: "FALLBACK" in RSRC or (pred_ml == labels).mean() >= (pred_lex == labels).mean())

**A3. Does it work for *everyone*? English vs Nigerian Pidgin.** Below are 12 English and 12 Pidgin sentences (hand-written for this lab, *illustrative and unvalidated*; for real research use a community-built dataset such as **NaijaSenti**, Muhammad et al., 2022). A prediction of `0` points (no lexicon word found) counts as wrong.

### 🔍 Code walkthrough — measuring a gap, not just an accuracy
`group_accuracy` scores the **same** classifier separately on English and Pidgin sentences. A bare difference of two accuracies hides how uncertain it is, so we also run **Fisher's exact test** on the 2×2 table (correct/wrong × group). A small *p* says the gap is unlikely to be chance; it does **not** say why the gap exists. That is the job of your analysis.

In [ ]:
#@title 🌍 Dialect gap
# Naija test sentences with labels and group (English or Pidgin)
NAIJA = [  # (text, label 1=positive 0=negative, group)
 ("The lecturer explained the topic very well and the class was excellent", 1, "English"),
 ("The network was fast and the app was easy to use", 1, "English"),
 ("I love this phone, the battery is great", 1, "English"),
 ("The cafeteria food was nice and the staff were helpful", 1, "English"),
 ("A wonderful library, clean and quiet", 1, "English"),
 ("Best data plan I have used, really good value", 1, "English"),
 ("The hostel is dirty and the water supply is terrible", 0, "English"),
 ("This bank app is slow and useless", 0, "English"),
 ("I am not happy with the delivery, it was very late", 0, "English"),
 ("The service was awful and the staff were rude", 0, "English"),
 ("The portal is broken again and I hate it", 0, "English"),
 ("The results were poor and disappointing", 0, "English"),
 ("This phone na correct thing, e dey work well", 1, "Pidgin"),
 ("The food sweet well well, I go buy again", 1, "Pidgin"),
 ("Dis lecturer dey teach well, I understand everything", 1, "Pidgin"),
 ("I like am, the data cheap and e dey fast", 1, "Pidgin"),
 ("E sweet me, the show na correct vibes", 1, "Pidgin"),
 ("Oga, you try, the work na fire", 1, "Pidgin"),
 ("Dis network don spoil again, e no dey work at all", 0, "Pidgin"),
 ("Dis service na scam, abeg make una fix am", 0, "Pidgin"),
 ("Na waste of money, the thing no good at all", 0, "Pidgin"),
 ("Wetin be dis, the app dey hang everyday, e don tire me", 0, "Pidgin"),
 ("Dis hostel na wahala, the water no dey and the room dirty", 0, "Pidgin"),
 ("Dis bank dey craze, dem don debit me twice", 0, "Pidgin")]
# Polarity from the lexicon: 1 positive, 0 negative, -1 abstain
def predict_polarity(text):
    # Score the sentence; abstain when the score is 0
    s = lex_score(text); return 1 if s > 0 else 0 if s < 0 else -1          # -1 = abstain (wrong)
# Accuracy for each language group
def group_accuracy():
    # Results dictionary
    out = {}
    # For English and for Pidgin
    for g in ["English", "Pidgin"]:
        # Pick that group's sentences
        rows = [(t, l) for t, l, gg in NAIJA if gg == g]
        # 1 if the prediction is right, else 0
        out[g] = [int(predict_polarity(t) == l) for t, l in rows]
    # Return results
    return out
# Run the group evaluation
acc_g = group_accuracy()
# Print accuracy for each group
for g, v in acc_g.items(): print(f"{g:<8} accuracy = {np.mean(v):.2f}  (n={len(v)})")
# 2x2 table: correct and wrong for each group
table = [[sum(acc_g["English"]), 12 - sum(acc_g["English"])], [sum(acc_g["Pidgin"]), 12 - sum(acc_g["Pidgin"])]]
# Fisher exact test: is the gap bigger than chance?
print("Fisher exact test English vs Pidgin: p =", f"{stats.fisher_exact(table)[1]:.4f}")

### Task A3 — Adapt the lexicon to Pidgin (and notice what is wrong with doing that here)
Add Pidgin words to `PIDGIN_LEXICON` (e.g. `sweet`, `correct`, `spoil`, `waste`, `wahala`…) with polarity scores. Re-measure. **Reflect:** you tuned on the same 24 sentences you test on — why is the improvement *inflated*, and how should you evaluate instead (hint: held-out data, community annotators)?

In [ ]:
# TODO(Student): add Pidgin words and polarity (−3 … +3)
# TODO: your Pidgin lexicon (start empty)
PIDGIN_LEXICON = {}
# Add your Pidgin words to the main lexicon
LEXICON.update(PIDGIN_LEXICON)
# Re-run the evaluation
acc_g2 = group_accuracy()
# Print the new accuracies
print({g: round(float(np.mean(v)), 2) for g, v in acc_g2.items()})

In [ ]:
#@title ✅ Auto-check for A3
check("Pidgin accuracy improved by ≥ 0.25 after adapting the lexicon", lambda: np.mean(acc_g2["Pidgin"]) - np.mean(acc_g["Pidgin"]) >= 0.25)
check("English accuracy did not collapse (≥ 0.8)",                  lambda: np.mean(acc_g2["English"]) >= 0.8)

## Part B — Spelling correction (word-processor feature)
Step 1: the **edit distance** between a typo and a dictionary word. Step 2: choose the most *frequent* word within one (then two) edits — a noisy-channel approximation.

In [ ]:
# TODO(Student): Levenshtein distance by dynamic programming
# TODO: edit distance between two words
def edit_distance(a, b):
    # Placeholder; replace with your dynamic programme
    return 0

In [ ]:
#@title ✅ Check + the corrector (error model = edits within distance 1, then 2; language model = word counts)
check("kitten → sitting = 3", lambda: edit_distance("kitten", "sitting") == 3)
check("'' → 'abc' = 3 and identical = 0", lambda: edit_distance("", "abc") == 3 and edit_distance("agent", "agent") == 0)

# Counter counts word frequencies
from collections import Counter
TEXT = """the agent perceives the environment and selects an action to maximise its performance this search algorithm uses a heuristic function to find the goal
a knowledge base stores facts and rules and the inference engine uses logic to derive new knowledge the probability of an event is a number between zero and one
natural language processing helps a computer understand a sentence and each word in the language the classifier is trained on an example set and tested on new data
the student writes a program to train a model and the training process improves the accuracy of the result an intelligent system must handle uncertainty and noise
this is a sample of the text used to count the words and how often each word appears in the course notes the lecture explains how to represent the problem as a state space"""
# Word frequencies from a text corpus; the alphabet
WORDS = Counter(re.findall(r"[a-z]+", TEXT.lower())); LETTERS = "abcdefghijklmnopqrstuvwxyz"
# All words one edit away from w
def edits1(w):
    # Split w at every position
    sp = [(w[:i], w[i:]) for i in range(len(w) + 1)]
    # Deletions, transpositions, replacements and insertions
    return ({a + b[1:] for a, b in sp if b} | {a + b[1] + b[0] + b[2:] for a, b in sp if len(b) > 1} |
            {a + c + b[1:] for a, b in sp if b for c in LETTERS} | {a + c + b for a, b in sp for c in LETTERS})
# Keep only known words
known = lambda ws: {w for w in ws if w in WORDS}
# Choose a correction
def correct(word):
    # Lower-case the word
    w = word.lower()
    # Prefer the word itself, then distance 1, then distance 2
    cands = known([w]) or known(edits1(w)) or known(e2 for e1 in edits1(w) for e2 in edits1(e1)) or {w}
    # Return the most frequent candidate
    return max(cands, key=lambda c: WORDS.get(c, 0))
# Test pairs: (misspelling, correct word)
PAIRS = [("sentnce", "sentence"), ("wrds", "words"), ("algoritm", "algorithm"), ("heuristc", "heuristic"), ("serach", "search"), ("agnet", "agent"),
         ("knowlege", "knowledge"), ("probabilty", "probability"), ("lanugage", "language"), ("classifer", "classifier"), ("trainng", "training"), ("enviroment", "environment")]
# Check each pair
hits = [correct(x) == y for x, y in PAIRS]; k = sum(hits)
# 95% Wilson CI of the accuracy
ci = stats.binomtest(k, len(hits)).proportion_ci(0.95, method="wilson")
# Print accuracy and CI
print(f"accuracy = {k}/{len(hits)} = {k/len(hits):.2f}  (95% CI [{ci.low:.2f}, {ci.high:.2f}])")
# Show the failures
for (x, y), h in zip(PAIRS, hits):
    # Print each failure
    if not h: print(f"  ✗ {x} → {correct(x)}  (wanted {y})")
check("corrector gets ≥ 10/12 right", lambda: k >= 10)

**Discuss.** Your corrector knows only ~100 words. What breaks with a *real* dictionary? (Real-word errors such as *form/from*, proper names, Yorùbá diacritics ẹ, ọ, ṣ.) How would a **language model** (context) fix *"I went form school"*?

## Part C — Machine translation (English → Yorùbá, toy)
Word-by-word translation fails because **word order and function words differ**. In Yorùbá an adjective follows its noun (*ìwé tuntun* "new book") and the definite marker *náà* comes last. Complete `reorder_np`, then compare **BLEU** with and without reordering.

> ⚠️ The lexicon below is a **teaching toy**. Please have a Yorùbá speaker review the tone marks before reusing it.

In [ ]:
#@title 🗣️ Toy rule-based MT
# English-to-Yoruba word list
LEX = {"i": "mo", "he": "ó", "she": "ó", "we": "a", "read": "kà", "see": "rí", "buy": "rà", "like": "fẹ́ràn",
       "book": "ìwé", "house": "ilé", "food": "oúnjẹ", "water": "omi", "child": "ọmọ", "new": "tuntun", "big": "ńlá", "the": "náà", "a": ""}
# Parts of speech
POS = {"new": "ADJ", "big": "ADJ", "the": "DET", "a": "DET", "book": "N", "house": "N", "food": "N", "water": "N", "child": "N"}

# Word-by-word baseline translator
def translate_word_by_word(sent):
    # Replace each word using the dictionary
    return " ".join(w for w in (LEX.get(t, t) for t in sent.lower().split()) if w)

def reorder_np(tokens):
    """Given English tokens of ONE noun phrase [DET] [ADJ] N → Yorùbá order: N ADJ [náà].  (e.g. the new book → ìwé tuntun náà)"""
    # Starter line: translates each token alone; replace it with the noun-phrase reordering rules
    return [LEX.get(t, t) for t in tokens]

# Rule-based translator that reorders noun phrases
def translate(sent):
    # Tokens and an output list
    toks = sent.lower().split(); out, i = [], 0
    # Process the sentence from left to right
    while i < len(toks):
        # Does a noun phrase begin here?
        if POS.get(toks[i]) in ("DET", "ADJ", "N"):               # collect the noun phrase
            # Mark the start
            j = i
            # Find the end of the noun phrase
            while j < len(toks) and POS.get(toks[j]) in ("DET", "ADJ", "N"): j += 1
            # Reorder it and add it to the output
            out += [w for w in reorder_np(toks[i:j]) if w]; i = j
        # Otherwise ...
        else:
            # ... translate the single word
            out.append(LEX.get(toks[i], toks[i])); i += 1
    # Join the words
    return " ".join(out)

In [ ]:
#@title 📏 BLEU and the comparison
# BLEU score: n-gram overlap between candidate and reference
def bleu(cand, ref, max_n=2):
    # Split into words
    c, r = cand.split(), ref.split()
    # Empty candidate scores 0
    if not c: return 0.0
    # Log-precision accumulator
    logp = 0.0
    # For n-grams of size 1 and 2
    for n in range(1, max_n + 1):
        # Count candidate and reference n-grams
        cn = Counter(tuple(c[i:i+n]) for i in range(len(c) - n + 1)); rn = Counter(tuple(r[i:i+n]) for i in range(len(r) - n + 1))
        # Clipped matches and total candidate n-grams
        clip = sum(min(v, rn[g]) for g, v in cn.items()); tot = max(sum(cn.values()), 1)
        # Add log precision (tiny value avoids log 0)
        logp += math.log(max(clip, 1e-9) / tot) / max_n                 # tiny smoothing so log(0) never happens
    # Brevity penalty for short candidates
    bp = min(1.0, math.exp(1 - len(r) / len(c)))
    # Final BLEU score
    return bp * math.exp(logp)

# Test sentences with reference translations
TESTS = [("I read the new book", "mo kà ìwé tuntun náà"), ("he buy a big house", "ó rà ilé ńlá"), ("we like the food", "a fẹ́ràn oúnjẹ náà"),
         ("I see the big house", "mo rí ilé ńlá náà"), ("she like new water", "ó fẹ́ràn omi tuntun")]
# BLEU for both translators
b_naive = [bleu(translate_word_by_word(s), ref) for s, ref in TESTS]; b_rule = [bleu(translate(s), ref) for s, ref in TESTS]
# Print each comparison
for (s, ref), a, b in zip(TESTS, b_naive, b_rule): print(f"{s:<22} naive: {translate_word_by_word(s):<24} rule: {translate(s):<24} ref: {ref}   BLEU {a:.2f} → {b:.2f}")
# Print the mean scores
print(f"\nmean BLEU-2  word-by-word = {np.mean(b_naive):.2f}   with reordering = {np.mean(b_rule):.2f}")
check("reordering moves the adjective after the noun", lambda: translate("the new book") == "ìwé tuntun náà")
check("reordering improves mean BLEU by ≥ 0.3",         lambda: np.mean(b_rule) - np.mean(b_naive) >= 0.3)

**Neural MT (optional, Colab with GPU/internet).** Set `RUN_NEURAL_MT = True` to try Meta's **NLLB-200** (covers Yorùbá `yor_Latn`, Hausa `hau_Latn`, Igbo `ibo_Latn`). Compare its output with your toy system and discuss **low-resource** issues (data scarcity, diacritics, code-switching).

In [ ]:
#@title 🌐 (Optional) Neural MT with NLLB-200 — disabled by default (large download)
# Switch for the heavy neural model (off by default)
RUN_NEURAL_MT = False
# Only run if switched on
if RUN_NEURAL_MT:
    # Try the real corpus first
    try:
        # Install the libraries
        ensure("transformers"); ensure("sentencepiece")
        # Import the model classes
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        # Pretrained NLLB translation model
        name = "facebook/nllb-200-distilled-600M"
        # Load tokenizer and model
        tok = AutoTokenizer.from_pretrained(name, src_lang="eng_Latn"); mdl = AutoModelForSeq2SeqLM.from_pretrained(name)
        # Tokenise an English sentence
        batch = tok(["I read the new book."], return_tensors="pt")
        # Translate into Yoruba
        out = mdl.generate(**batch, forced_bos_token_id=tok.convert_tokens_to_ids("yor_Latn"), max_new_tokens=40)
        # Print the translation
        print(tok.batch_decode(out, skip_special_tokens=True))
    # If it is unavailable ...
    except Exception as e:
        # Log if it fails
        log.warning("neural MT skipped: %s", e)
# Otherwise ...
else:
    # Tell the student how to enable it
    print("Neural MT disabled (set RUN_NEURAL_MT = True in Colab to try it).")

## Part D — Speech recognition (isolated words) with spectrograms and DTW
A recogniser turns sound into text. The classical pipeline: **waveform → short-time spectrum (features) → align to templates (DTW) → pick the nearest word.** We synthesise five "words" (three-tone melodies), distort their **timing** and add **noise**, then recognise them. *(Real systems replace templates by neural acoustic models, e.g. Whisper, but the same ideas — features, alignment, noise-robustness — remain.)*

### 🔍 Code walkthrough — from sound to features
Each synthetic word is a sequence of three tones. `features` cuts the signal into short overlapping frames, takes the magnitude of the Fourier transform of each, pools it into 16 frequency bands, and normalises. The result is a small matrix (time × band) that **ignores loudness** and keeps the pattern. Two utterances of the same word then look similar even when one is spoken more slowly. Dynamic time warping, next, is the tool that lines them up.

In [ ]:
#@title 🎙️ Synthetic words → log-band spectrogram features
# Sampling rate: 8000 samples per second
SR = 8000
# Each word is a sequence of three tones (Hz)
WORDS_HZ = {"yes": [300, 600, 900], "no": [900, 600, 300], "stop": [400, 400, 800], "go": [800, 300, 300], "help": [500, 900, 500]}

# Create a synthetic spoken word
def synth(word, rng, stretch=True, snr_db=None):
    # List of audio segments
    segs = []
    # One segment per tone
    for f in WORDS_HZ[word]:
        # Random duration to simulate different speaking rates
        dur = 0.15 * (rng.uniform(0.7, 1.4) if stretch else 1.0)               # speaking-rate variation
        # Generate a sine wave with a smooth envelope
        t = np.arange(int(SR * dur)) / SR; segs.append(np.sin(2 * np.pi * f * t) * np.hanning(len(t)))
    # Join the segments
    x = np.concatenate(segs)
    # Add noise at a chosen signal-to-noise ratio
    if snr_db is not None:
        # Scale Gaussian noise to the target SNR
        noise = rng.normal(0, 1, len(x)); p = np.mean(x ** 2) / (10 ** (snr_db / 10)); x = x + noise * math.sqrt(p / np.mean(noise ** 2))
    # Return the audio
    return x

# Convert audio to a feature sequence
def features(x, n_fft=256, hop=128, bands=16):
    # Slice into overlapping windowed frames
    win = np.hanning(n_fft); frames = [x[i:i + n_fft] * win for i in range(0, len(x) - n_fft, hop)]
    # Magnitude spectrum, up to 4 kHz
    mag = np.abs(np.fft.rfft(np.array(frames), axis=1))[:, :128]               # 0 … 4 kHz
    # Pool into 16 frequency bands, log scale
    F = np.log1p(mag.reshape(len(frames), bands, -1).sum(axis=2))               # 16 equal bands
    # Normalise each band
    return (F - F.mean(0)) / (F.std(0) + 1e-6)                                  # per-utterance normalisation

In [ ]:
# TODO(Student): dynamic time warping distance between feature sequences A (T1×d) and B (T2×d)
# TODO: dynamic time warping distance
def dtw(A, B):
    # Placeholder; replace with your DTW
    return 0.0

In [ ]:
#@title ✅ Check DTW, then recognise at different noise levels (40 trials per word per SNR)
# Seeded generator
rng0 = np.random.default_rng(SEED)
# One template per word
TEMPLATES = {w: features(synth(w, rng0, stretch=False)) for w in WORDS_HZ}
# Two example words to compare
a = features(synth("yes", np.random.default_rng(1))); b = features(synth("no", np.random.default_rng(2)))
check("DTW(x, x) = 0",                           lambda: abs(dtw(a, a)) < 1e-9)
check("DTW is symmetric",                        lambda: abs(dtw(a, b) - dtw(b, a)) < 1e-9)
check("a time-stretched 'yes' is closer to the 'yes' template than 'no' is", lambda: dtw(a, TEMPLATES["yes"]) < dtw(b, TEMPLATES["yes"]))

# Recognise = nearest template by DTW distance
def recognise(x): f = features(x); return min(TEMPLATES, key=lambda w: dtw(f, TEMPLATES[w]))
# Results by noise level
results = {}
# Try five noise levels
for snr in [20, 10, 5, 0, -5]:
    # Seeded generator; list of outcomes
    rng = np.random.default_rng(SEED + snr + 100); ok = []
    # For each word
    for w in WORDS_HZ:
        # 40 noisy tests per word
        for _ in range(40): ok.append(recognise(synth(w, rng, snr_db=snr)) == w)
    # Accuracy and Wilson CI
    k = sum(ok); ci = stats.binomtest(k, len(ok)).proportion_ci(0.95, method="wilson"); results[snr] = (k / len(ok), ci.low, ci.high)
    # Print the result
    print(f"SNR {snr:>3} dB: accuracy = {k/len(ok):.3f}  95% CI [{ci.low:.3f}, {ci.high:.3f}]  (n={len(ok)})")
# Plotting library
import matplotlib.pyplot as plt
# Plot accuracy with error bars
plt.figure(figsize=(5, 3)); s = sorted(results); plt.errorbar(s, [results[x][0] for x in s], yerr=[[results[x][0]-results[x][1] for x in s], [results[x][2]-results[x][0] for x in s]], fmt="o-")
# Labels and display
plt.xlabel("SNR (dB)"); plt.ylabel("word accuracy"); plt.ylim(0, 1.05); plt.title("Recogniser accuracy vs noise"); plt.tight_layout(); plt.show()
check("≥ 95% accurate at 20 dB SNR", lambda: results[20][0] >= 0.95)
check("accuracy falls as noise rises (−5 dB < 20 dB)", lambda: results[-5][0] < results[20][0])

**Discuss.** (1) Why does DTW beat a plain frame-by-frame Euclidean distance when speakers talk at different speeds? (2) Which realistic problems does our toy ignore (speaker variation, accents, vocabulary size, continuous speech)? (3) For Yorùbá, **tone** carries meaning — what feature would you add?

## 📝 Assignment (5 marks, individual, formative)
1. *(2)* Write **10 new labelled sentences** in English and Pidgin (or Yorùbá). Report the accuracy of each group, the gap and its **bootstrap 95% CI**.
2. *(2)* Improve the spelling corrector with at least three error types and test it on **20 word pairs**: accuracy with Wilson CI.
3. *(1)* In ≤120 words, explain two reasons why BLEU can mislead for a low-resource language.

*Marked with the course rubric (class note, Table 0.3): correctness, evidence (numbers with CI and a named test), code clarity, interpretation. Submit the **executed notebook** (Runtime → Run all, outputs visible) as a Colab share link or `.ipynb` before the next lab.*

## Reflection
- Which of the four NLP applications you built today would you trust most for Nigerian users, and which least? Justify with your measurements.
- Name two ways *data* (not algorithms) limit NLP for African languages.

```text
Write your reflection here…
```

## Extensions
- Replace your lexicon by a **Naive Bayes** classifier trained on the 24 sentences; evaluate with leave-one-out CV and compare English vs Pidgin accuracy.
- Add **Damerau–Levenshtein** (transpositions cost 1) and a bigram language model to fix *real-word* errors.
- Record yourself saying two words; extract features with `librosa` and test DTW on real audio.

In [ ]:
checks_summary()  # how many auto-checks have you passed?